In [99]:
CONFIDENCE_THRESHOLD = 0.20

In [3]:
import os
import cv2
import json
import numpy as np
import onnxruntime as ort

from pathlib import Path

In [ ]:
# =========================
# PATHS
# =========================

DET_MODEL_PATH = "Det_Retina_Net.onnx"
REC_MODEL_PATH = "w600k_mbf.onnx"

# =========================
# DETECTOR
# =========================

DET_INPUT_SIZE = (640, 640)

SCORE_THRESHOLD = 0.50
NMS_THRESHOLD = 0.40

STRIDES = [8, 16, 32]

# =========================
# FACE ALIGNMENT
# =========================

ARCFACE_DST = np.array(
    [
        [38.2946, 51.6963],
        [73.5318, 51.5014],
        [56.0252, 71.7366],
        [41.5493, 92.3655],
        [70.7299, 92.2041]
    ],
    dtype=np.float32
)

FACE_SIZE = 112

In [5]:
def letterbox(
    image,
    new_shape=(640,640),
    color=(0,0,0)
):
    
    h, w = image.shape[:2]

    scale = min(
        new_shape[0] / h,
        new_shape[1] / w
    )

    nw = int(round(w * scale))
    nh = int(round(h * scale))

    resized = cv2.resize(
        image,
        (nw, nh),
        interpolation=cv2.INTER_LINEAR
    )

    canvas = np.full(
        (
            new_shape[0],
            new_shape[1],
            3
        ),
        color,
        dtype=np.uint8
    )

    pad_x = (new_shape[1] - nw) // 2
    pad_y = (new_shape[0] - nh) // 2

    canvas[
        pad_y:pad_y+nh,
        pad_x:pad_x+nw
    ] = resized

    return canvas, scale, pad_x, pad_y

In [6]:
def preprocess_detector(
    image
):

    img_lb, scale, pad_x, pad_y = letterbox(
        image,
        DET_INPUT_SIZE
    )

    rgb = cv2.cvtColor(
        img_lb,
        cv2.COLOR_BGR2RGB
    )

    blob = rgb.astype(np.float32)

    blob = (blob - 127.5) / 128.0

    blob = np.transpose(
        blob,
        (2,0,1)
    )

    blob = np.expand_dims(
        blob,
        axis=0
    )

    return (
        blob,
        scale,
        pad_x,
        pad_y,
        img_lb
    )

In [7]:
def generate_centers(
    fm_h,
    fm_w,
    stride,
    num_anchors=2
):

    ys, xs = np.mgrid[
        :fm_h,
        :fm_w
    ]

    centers = np.stack(
        [xs, ys],
        axis=-1
    ).astype(np.float32)

    centers = (
        centers + 0.5
    ) * stride

    centers = centers.reshape(-1,2)

    centers = np.repeat(
        centers,
        num_anchors,
        axis=0
    )

    return centers

In [8]:
def distance2bbox(
    points,
    distance,
    stride
):

    x1 = points[:,0] - distance[:,0] * stride
    y1 = points[:,1] - distance[:,1] * stride

    x2 = points[:,0] + distance[:,2] * stride
    y2 = points[:,1] + distance[:,3] * stride

    return np.stack(
        [x1,y1,x2,y2],
        axis=1
    )


def distance2kps(
    points,
    preds,
    stride
):

    preds = preds.reshape(
        -1,
        5,
        2
    )

    kps = np.zeros_like(preds)

    kps[:,:,0] = (
        points[:,None,0]
        + preds[:,:,0] * stride
    )

    kps[:,:,1] = (
        points[:,None,1]
        + preds[:,:,1] * stride
    )

    return kps

In [9]:
class SCRFDDetector:

    def __init__(
        self,
        model_path
    ):

        self.session = ort.InferenceSession(
            model_path,
            providers=["CPUExecutionProvider"]
        )

        self.input_name = (
            self.session.get_inputs()[0].name
        )

    def decode(
        self,
        outputs,
        score_thresh=SCORE_THRESHOLD
    ):

        scores_list = outputs[:3]
        bbox_list = outputs[3:6]
        kps_list = outputs[6:9]

        all_boxes = []
        all_scores = []
        all_kps = []

        for score_pred, bbox_pred, kps_pred, stride in zip(
            scores_list,
            bbox_list,
            kps_list,
            STRIDES
        ):

            fm_h = DET_INPUT_SIZE[0] // stride
            fm_w = DET_INPUT_SIZE[1] // stride

            centers = generate_centers(
                fm_h,
                fm_w,
                stride
            )

            scores = score_pred[:,0]

            keep = scores > score_thresh

            if keep.sum() == 0:
                continue

            centers = centers[keep]

            scores = scores[keep]

            bbox_pred = bbox_pred[keep]

            kps_pred = kps_pred[keep]

            boxes = distance2bbox(
                centers,
                bbox_pred,
                stride
            )

            kps = distance2kps(
                centers,
                kps_pred,
                stride
            )

            all_boxes.append(boxes)
            all_scores.append(scores)
            all_kps.append(kps)

        if len(all_boxes) == 0:
            return [], [], []

        boxes = np.concatenate(all_boxes)

        scores = np.concatenate(all_scores)

        kps = np.concatenate(all_kps)

        return boxes, scores, kps

    def nms(
        self,
        boxes,
        scores,
        kps
    ):

        bboxes = []

        for box in boxes:

            x1,y1,x2,y2 = box

            bboxes.append(
                [
                    float(x1),
                    float(y1),
                    float(x2-x1),
                    float(y2-y1)
                ]
            )

        idxs = cv2.dnn.NMSBoxes(
            bboxes,
            scores.tolist(),
            SCORE_THRESHOLD,
            NMS_THRESHOLD
        )

        if len(idxs) == 0:
            return [], [], []

        idxs = idxs.flatten()

        return (
            boxes[idxs],
            scores[idxs],
            kps[idxs]
        )

    def detect(
        self,
        image
    ):

        blob, scale, pad_x, pad_y, _ = preprocess_detector(
            image
        )

        outputs = self.session.run(
            None,
            {
                self.input_name: blob
            }
        )

        boxes, scores, kps = self.decode(
            outputs
        )

        if len(boxes) == 0:
            return [], [], []

        boxes, scores, kps = self.nms(
            boxes,
            scores,
            kps
        )

        boxes[:,[0,2]] -= pad_x
        boxes[:,[1,3]] -= pad_y

        kps[:,:,0] -= pad_x
        kps[:,:,1] -= pad_y

        boxes /= scale
        kps /= scale

        return (
            boxes,
            scores,
            kps
        )

In [10]:
def align_face(
    image,
    landmarks
):

    src = landmarks.astype(
        np.float32
    )

    M, _ = cv2.estimateAffinePartial2D(
        src,
        ARCFACE_DST,
        method=cv2.LMEDS
    )

    aligned = cv2.warpAffine(
        image,
        M,
        (FACE_SIZE, FACE_SIZE),
        borderValue=0
    )

    return aligned

In [11]:
class MobileFaceNetExtractor:

    def __init__(
        self,
        model_path
    ):

        self.session = ort.InferenceSession(
            model_path,
            providers=["CPUExecutionProvider"]
        )

        self.input_name = (
            self.session.get_inputs()[0].name
        )

    def preprocess(
        self,
        face
    ):

        rgb = cv2.cvtColor(
            face,
            cv2.COLOR_BGR2RGB
        )

        blob = rgb.astype(
            np.float32
        )

        blob = (
            blob - 127.5
        ) / 128.0

        blob = np.transpose(
            blob,
            (2,0,1)
        )

        blob = np.expand_dims(
            blob,
            axis=0
        )

        return blob

    def get_embedding(
        self,
        aligned_face
    ):

        blob = self.preprocess(
            aligned_face
        )

        emb = self.session.run(
            None,
            {
                self.input_name: blob
            }
        )[0][0]

        emb = emb / np.linalg.norm(emb)

        return emb.astype(
            np.float32
        )

In [12]:
detector = SCRFDDetector(
    DET_MODEL_PATH
)

extractor = MobileFaceNetExtractor(
    REC_MODEL_PATH
)

print("Models Loaded")

Models Loaded


In [16]:
image = cv2.imread(
    "12-03-2026.jpeg"
)

boxes, scores, kps = detector.detect(
    image
)

print(
    "Faces Found:",
    len(boxes)
)

if len(boxes) > 0:

    largest_idx = np.argmax(
        (boxes[:,2]-boxes[:,0]) *
        (boxes[:,3]-boxes[:,1])
    )

    aligned = align_face(
        image,
        kps[largest_idx]
    )

    embedding = extractor.get_embedding(
        aligned
    )

    print(
        "Embedding Shape:",
        embedding.shape
    )

    cv2.imwrite(
        "aligned_face.jpg",
        aligned
    )

Faces Found: 6
Embedding Shape: (512,)


In [27]:
! python -m pip install tqdm
! python -m pip show tqdm


  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.68.3-py3-none-any.whl (78 kB)
Name: tqdm
Version: 4.68.3
Summary: Fast, Extensible Progress Meter
Home-page: https://tqdm.github.io
Author: 
Author-email: 
License: MPL-2.0 AND MIT
Location: c:\Users\ravip\AppData\Local\Programs\Python\Python314\Lib\site-packages
Requires: colorama
Required-by: 


In [28]:
import os
import cv2
import json
import numpy as np

from pathlib import Path
from tqdm import tqdm

In [29]:
STUDENT_DATASET_DIR = r"DataSet/students"

ARTIFACT_DIR = "artifacts"

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True
)

In [30]:
VALID_EXTENSIONS = [
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
]

In [31]:
student_folders = []

for item in os.listdir(STUDENT_DATASET_DIR):

    full_path = os.path.join(
        STUDENT_DATASET_DIR,
        item
    )

    if os.path.isdir(full_path):

        if item.startswith("student_"):

            student_folders.append(item)

student_folders = sorted(student_folders)

print("Students Found:", len(student_folders))
print(student_folders)

Students Found: 9
['student_01', 'student_02', 'student_03', 'student_04', 'student_05', 'student_06', 'student_07', 'student_08', 'student_09']


In [32]:
label_map = {}

reverse_label_map = {}

for idx, folder_name in enumerate(student_folders):

    roll_no = folder_name.replace(
        "student_",
        ""
    )

    label_map[roll_no] = idx

    reverse_label_map[str(idx)] = roll_no

print(label_map)

{'01': 0, '02': 1, '03': 2, '04': 3, '05': 4, '06': 5, '07': 6, '08': 7, '09': 8}


In [33]:
with open(
    os.path.join(
        ARTIFACT_DIR,
        "label_map.json"
    ),
    "w"
) as f:

    json.dump(
        label_map,
        f,
        indent=4
    )


with open(
    os.path.join(
        ARTIFACT_DIR,
        "reverse_label_map.json"
    ),
    "w"
) as f:

    json.dump(
        reverse_label_map,
        f,
        indent=4
    )

print("Label Maps Saved")

Label Maps Saved


In [34]:
def get_largest_face_index(
    boxes
):

    if len(boxes) == 0:
        return None

    areas = (
        boxes[:,2] - boxes[:,0]
    ) * (
        boxes[:,3] - boxes[:,1]
    )

    return np.argmax(
        areas
    )

In [35]:
def image_to_embedding(
    image_path
):

    image = cv2.imread(
        image_path
    )

    if image is None:
        return None

    boxes, scores, kps = detector.detect(
        image
    )

    if len(boxes) == 0:
        return None

    idx = get_largest_face_index(
        boxes
    )

    aligned = align_face(
        image,
        kps[idx]
    )

    embedding = extractor.get_embedding(
        aligned
    )

    return embedding

In [36]:
embeddings = []

labels = []

failed_images = []

total_images = 0

In [37]:
for folder_name in student_folders:

    roll_no = folder_name.replace(
        "student_",
        ""
    )

    label = label_map[
        roll_no
    ]

    student_dir = os.path.join(
        STUDENT_DATASET_DIR,
        folder_name
    )

    image_files = []

    for file_name in os.listdir(student_dir):

        ext = Path(
            file_name
        ).suffix.lower()

        if ext in VALID_EXTENSIONS:

            image_files.append(
                file_name
            )

    print(
        f"\nProcessing Student {roll_no}"
    )

    for image_name in tqdm(image_files):

        image_path = os.path.join(
            student_dir,
            image_name
        )

        total_images += 1

        embedding = image_to_embedding(
            image_path
        )

        if embedding is None:

            failed_images.append(
                image_path
            )

            continue

        embeddings.append(
            embedding
        )

        labels.append(
            label
        )


Processing Student 01


100%|██████████| 5/5 [00:00<00:00, 12.65it/s]



Processing Student 02


100%|██████████| 5/5 [00:00<00:00, 11.57it/s]



Processing Student 03


...

100%|██████████| 5/5 [00:00<00:00, 12.10it/s]



Processing Student 08


100%|██████████| 5/5 [00:00<00:00, 12.85it/s]



Processing Student 09


100%|██████████| 5/5 [00:00<00:00, 10.29it/s]


In [38]:
embeddings = np.array(
    embeddings,
    dtype=np.float32
)

labels = np.array(
    labels,
    dtype=np.int64
)

print(
    "Embeddings Shape:",
    embeddings.shape
)

print(
    "Labels Shape:",
    labels.shape
)

print(
    "Failed Images:",
    len(failed_images)
)

Embeddings Shape: (45, 512)
Labels Shape: (45,)
Failed Images: 0


In [39]:
np.save(
    os.path.join(
        ARTIFACT_DIR,
        "embeddings.npy"
    ),
    embeddings
)

np.save(
    os.path.join(
        ARTIFACT_DIR,
        "labels.npy"
    ),
    labels
)

print("Dataset Saved")

Dataset Saved


In [40]:
with open(
    os.path.join(
        ARTIFACT_DIR,
        "failed_images.txt"
    ),
    "w"
) as f:

    for item in failed_images:

        f.write(
            item + "\n"
        )

print(
    "Failed image list saved"
)

Failed image list saved


In [41]:
for roll_no, idx in label_map.items():

    count = np.sum(
        labels == idx
    )

    print(
        f"Roll {roll_no} : {count}"
    )

Roll 01 : 5
Roll 02 : 5
Roll 03 : 5
Roll 04 : 5
Roll 05 : 5
Roll 06 : 5
Roll 07 : 5
Roll 08 : 5
Roll 09 : 5


In [42]:
print(
    embeddings[0][:10]
)

print(
    labels[0]
)

[ 0.00146713  0.0148715   0.02196483  0.01078529 -0.04126714 -0.01655331
  0.05642366  0.03747728  0.02918855 -0.02563542]
0


In [44]:
! python -m pip install torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ----------------------------

In [45]:
import os
import json
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

In [46]:
ARTIFACT_DIR = "artifacts"

BATCH_SIZE = 32

EPOCHS = 50

LEARNING_RATE = 1e-3

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("DEVICE:", DEVICE)

DEVICE: cpu


In [47]:
embeddings = np.load(
    os.path.join(
        ARTIFACT_DIR,
        "embeddings.npy"
    )
)

labels = np.load(
    os.path.join(
        ARTIFACT_DIR,
        "labels.npy"
    )
)

print(
    "Embeddings:",
    embeddings.shape
)

print(
    "Labels:",
    labels.shape
)

Embeddings: (45, 512)
Labels: (45,)


In [48]:
with open(
    os.path.join(
        ARTIFACT_DIR,
        "label_map.json"
    ),
    "r"
) as f:

    label_map = json.load(f)

NUM_CLASSES = len(
    label_map
)

print(
    "NUM_CLASSES:",
    NUM_CLASSES
)

NUM_CLASSES: 9


In [49]:
class EmbeddingDataset(
    Dataset
):

    def __init__(
        self,
        embeddings,
        labels
    ):

        self.X = torch.tensor(
            embeddings,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx]
        )

In [50]:
dataset = EmbeddingDataset(
    embeddings,
    labels
)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(
    "Training Samples:",
    len(dataset)
)

Training Samples: 45


In [61]:
# class AttendanceClassifier(
#     nn.Module
# ):

#     def __init__(
#         self,
#         num_classes
#     ):

#         super().__init__()

#         self.net = nn.Sequential(

#             nn.Linear(
#                 512,
#                 256
#             ),

#             nn.ReLU(),

#             nn.Dropout(
#                 0.3
#             ),

#             nn.Linear(
#                 256,
#                 128
#             ),

#             nn.ReLU(),

#             nn.Dropout(
#                 0.3
#             ),

#             nn.Linear(
#                 128,
#                 num_classes
#             )
#         )

#     def forward(
#         self,
#         x
#     ):

#         return self.net(x)
import torch.nn as nn

class AttendanceClassifier(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.fc1 = nn.Linear(512, 128)

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(
            128,
            num_classes
        )

    def forward(self, x):

        x = self.fc1(x)

        x = self.relu(x)

        x = self.fc2(x)

        return x

In [62]:
model = AttendanceClassifier(
    NUM_CLASSES
)

model = model.to(
    DEVICE
)

print(model)

AttendanceClassifier(
  (fc1): Linear(in_features=512, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=9, bias=True)
)


In [63]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [64]:
for epoch in range(EPOCHS):

    model.train()

    epoch_loss = 0.0

    correct = 0

    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(
            DEVICE
        )

        y_batch = y_batch.to(
            DEVICE
        )

        optimizer.zero_grad()

        outputs = model(
            X_batch
        )

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        epoch_loss += (
            loss.item()
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            preds == y_batch
        ).sum().item()

        total += len(
            y_batch
        )

    acc = (
        correct / total
    ) * 100

    print(
        f"Epoch {epoch+1:03d}/{EPOCHS} "
        f"Loss={epoch_loss:.4f} "
        f"Acc={acc:.2f}%"
    )

Epoch 001/50 Loss=4.4037 Acc=11.11%
Epoch 002/50 Loss=4.3390 Acc=28.89%
Epoch 003/50 Loss=4.2883 Acc=42.22%
Epoch 004/50 Loss=4.2321 Acc=46.67%
Epoch 005/50 Loss=4.1885 Acc=66.67%
Epoch 006/50 Loss=4.1380 Acc=82.22%
Epoch 007/50 Loss=4.0716 Acc=84.44%
Epoch 008/50 Loss=4.0196 Acc=88.89%
Epoch 009/50 Loss=3.9286 Acc=93.33%
Epoch 010/50 Loss=3.8759 Acc=95.56%
Epoch 011/50 Loss=3.8033 Acc=95.56%
Epoch 012/50 Loss=3.6876 Acc=100.00%
Epoch 013/50 Loss=3.6029 Acc=100.00%
Epoch 014/50 Loss=3.5508 Acc=100.00%
Epoch 015/50 Loss=3.4758 Acc=100.00%
Epoch 016/50 Loss=3.3173 Acc=100.00%
Epoch 017/50 Loss=3.2540 Acc=100.00%
Epoch 018/50 Loss=3.1607 Acc=100.00%
Epoch 019/50 Loss=3.0704 Acc=100.00%
Epoch 020/50 Loss=2.9133 Acc=100.00%
Epoch 021/50 Loss=2.8199 Acc=100.00%
Epoch 022/50 Loss=2.7406 Acc=100.00%
Epoch 023/50 Loss=2.6928 Acc=100.00%
Epoch 024/50 Loss=2.4543 Acc=100.00%
Epoch 025/50 Loss=2.4003 Acc=100.00%
Epoch 026/50 Loss=2.2194 Acc=100.00%
Epoch 027/50 Loss=2.1592 Acc=100.00%
Epoch 028/50

In [65]:
torch.save(
    model.state_dict(),
    os.path.join(
        ARTIFACT_DIR,
        "attendance_classifier.pth"
    )
)

print(
    "PyTorch Model Saved"
)

PyTorch Model Saved


In [66]:
model.eval()

with torch.no_grad():

    X = torch.tensor(
        embeddings,
        dtype=torch.float32
    ).to(DEVICE)

    y = torch.tensor(
        labels,
        dtype=torch.long
    ).to(DEVICE)

    outputs = model(X)

    preds = torch.argmax(
        outputs,
        dim=1
    )

    acc = (
        (preds == y)
        .float()
        .mean()
        .item()
    )

print(
    f"Training Accuracy = {acc*100:.2f}%"
)

Training Accuracy = 100.00%


In [67]:
cpu_model = AttendanceClassifier(
    NUM_CLASSES
)

cpu_model.load_state_dict(
    torch.load(
        os.path.join(
            ARTIFACT_DIR,
            "attendance_classifier.pth"
        ),
        map_location="cpu"
    )
)

cpu_model.eval()

AttendanceClassifier(
  (fc1): Linear(in_features=512, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=9, bias=True)
)

In [68]:
dummy_input = torch.randn(
    1,
    512
)

In [83]:
! python -m pip install pandas

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - ------------------------------

In [69]:
torch.onnx.export(
    cpu_model,

    dummy_input,

    os.path.join(
        ARTIFACT_DIR,
        "attendance_classifier.onnx"
    ),

    input_names=[
        "embedding"
    ],

    output_names=[
        "scores"
    ],

    dynamic_axes={
        "embedding": {
            0: "batch_size"
        },
        "scores": {
            0: "batch_size"
        }
    },

    opset_version=11
)

print(
    "ONNX Export Complete"
)

C:\Users\ravip\AppData\Local\Temp\ipykernel_12372\910664867.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0623 16:14:50.429000 12372 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0623 16:14:51.274000 12372 site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W0623 16:14:51.277000 12372 sit

[torch.onnx] Obtain model graph for `AttendanceClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AttendanceClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


c:\Users\ravip\AppData\Local\Programs\Python\Python314\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\ravip\AppData\Local\Programs\Python\Python314\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\ravip\AppData\Local\Programs\Python\Python314\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX Export Complete


In [70]:
import onnxruntime as ort

session = ort.InferenceSession(
    os.path.join(
        ARTIFACT_DIR,
        "attendance_classifier.onnx"
    ),
    providers=[
        "CPUExecutionProvider"
    ]
)

print(
    "Input:",
    session.get_inputs()[0].shape
)

print(
    "Output:",
    session.get_outputs()[0].shape
)

Input: ['batch_size', 512]
Output: ['batch_size', 9]


In [100]:
p=10
sample = embeddings[p]

sample = np.expand_dims(
    sample,
    axis=0
).astype(np.float32)

scores = session.run(
    None,
    {
        "embedding": sample
    }
)[0]

pred = np.argmax(
    scores,
    axis=1
)[0]

print(
    "Predicted Class:",
    pred
)

print(
    "Actual Class:",
    labels[p]
)

Predicted Class: 2
Actual Class: 2


In [101]:
import os
import cv2
import json
import numpy as np
import pandas as pd
import onnxruntime as ort

In [102]:
GROUP_PHOTO_DIR = r"DataSet/group_photos"

ARTIFACT_DIR = "artifacts"

CSV_OUTPUT = "attendance.csv"

In [103]:
with open(
    os.path.join(
        ARTIFACT_DIR,
        "label_map.json"
    ),
    "r"
) as f:

    label_map = json.load(f)

with open(
    os.path.join(
        ARTIFACT_DIR,
        "reverse_label_map.json"
    ),
    "r"
) as f:

    reverse_label_map = json.load(f)

print(
    "Students:",
    len(label_map)
)

Students: 9


In [104]:
classifier_session = ort.InferenceSession(
    os.path.join(
        ARTIFACT_DIR,
        "attendance_classifier.onnx"
    ),
    providers=[
        "CPUExecutionProvider"
    ]
)

classifier_input_name = (
    classifier_session
    .get_inputs()[0]
    .name
)

print(
    "Classifier Loaded"
)

Classifier Loaded


In [105]:
def softmax(x):

    x = x - np.max(
        x,
        axis=1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return (
        exp_x
        /
        np.sum(
            exp_x,
            axis=1,
            keepdims=True
        )
    )

In [106]:
def predict_student(
    embedding
):

    embedding = np.expand_dims(
        embedding,
        axis=0
    ).astype(np.float32)

    scores = classifier_session.run(
        None,
        {
            classifier_input_name:
            embedding
        }
    )[0]

    probs = softmax(
        scores
    )

    pred_idx = int(
        np.argmax(
            probs,
            axis=1
        )[0]
    )

    confidence = float(
        probs[0][pred_idx]
    )

    roll_no = reverse_label_map[
        str(pred_idx)
    ]

    return (
        roll_no,
        confidence
    )

In [107]:
ATTENDANCE_FACE_DIR = "AttendanceFaces"

os.makedirs(
    ATTENDANCE_FACE_DIR,
    exist_ok=True
)

# CONFIDENCE_THRESHOLD = 0.70

In [108]:
def process_group_photo(
    image_path
):

    image = cv2.imread(
        image_path
    )

    if image is None:
        raise ValueError(
            f"Cannot read {image_path}"
        )

    boxes, scores, kps = detector.detect(
        image
    )

    attendance = {}

    for landmark in kps:

        try:

            aligned_face = align_face(
                image,
                landmark
            )

            embedding = (
                extractor
                .get_embedding(
                    aligned_face
                )
            )

            roll_no, confidence = (
                predict_student(
                    embedding
                )
            )

            if confidence < CONFIDENCE_THRESHOLD:
                continue

            if roll_no not in attendance:

                attendance[
                    roll_no
                ] = {
                    "confidence": confidence,
                    "face": aligned_face
                }

            else:

                old_conf = attendance[
                    roll_no
                ]["confidence"]

                if confidence > old_conf:

                    attendance[
                        roll_no
                    ] = {
                        "confidence": confidence,
                        "face": aligned_face
                    }

        except Exception as e:

            print(
                "Face Error:",
                e
            )

    return attendance

In [109]:
def save_attendance_faces(
    date_str,
    attendance
):

    save_dir = os.path.join(
        ATTENDANCE_FACE_DIR,
        date_str
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    for roll_no, info in attendance.items():

        confidence = info[
            "confidence"
        ]

        face = info[
            "face"
        ]

        filename = (
            f"{roll_no}_{confidence:.2f}.jpg"
        )

        save_path = os.path.join(
            save_dir,
            filename
        )

        cv2.imwrite(
            save_path,
            face
        )

In [110]:
roll_numbers = sorted(
    label_map.keys()
)

columns = [
    "Date",
    "TotalStudents"
]

for roll in roll_numbers:

    columns.append(
        roll
    )

    columns.append(
        f"{roll}_conf"
    )

print(columns)

['Date', 'TotalStudents', '01', '01_conf', '02', '02_conf', '03', '03_conf', '04', '04_conf', '05', '05_conf', '06', '06_conf', '07', '07_conf', '08', '08_conf', '09', '09_conf']


In [ ]:
VALID_EXTENSIONS = [
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
]

group_images = []

for file_name in os.listdir(
    GROUP_PHOTO_DIR
):

    ext = os.path.splitext(
        file_name
    )[1].lower()

    if ext in VALID_EXTENSIONS:

        group_images.append(
            file_name
        )

group_images = sorted(
    group_images
)

print(
    "Group Photos:",
    len(group_images)
)

Group Photos: 11


In [112]:
attendance_rows = []

for image_name in group_images:

    print(
        "\nProcessing:",
        image_name
    )

    image_path = os.path.join(
        GROUP_PHOTO_DIR,
        image_name
    )

    attendance = process_group_photo(
        image_path
    )

    date_str = os.path.splitext(
        image_name
    )[0]

    save_attendance_faces(
        date_str,
        attendance
    )

    row = {}

    row["Date"] = date_str

    row["TotalStudents"] = len(
        attendance
    )

    for roll in roll_numbers:

        row[roll] = 0

        row[f"{roll}_conf"] = 0.0

    for roll_no, info in attendance.items():

        row[roll_no] = 1

        row[
            f"{roll_no}_conf"
        ] = round(
            info["confidence"],
            4
        )

    attendance_rows.append(
        row
    )

    print(
        f"Present: {len(attendance)}"
    )


Processing: 12-03-2026.jpeg
Present: 5

Processing: 13-02-2026.jpeg
Present: 4

Processing: 14-02-2026.jpeg
Present: 2

Processing: 15-02-2026.jpeg
Present: 4

Processing: 16-02-2026.jpeg
Present: 4

Processing: 17-03-2026.jpeg
Present: 4

Processing: 18-03-2026.jpeg
Present: 4

Processing: 19-03-2026.jpeg
Present: 2

Processing: 20-03-2026.jpeg
Present: 2

Processing: 23-03-2026.jpeg
Present: 2

Processing: 25-03-2026.jpeg
Present: 2


In [113]:
attendance_df = pd.DataFrame(
    attendance_rows
)

attendance_df = attendance_df[
    columns
]

attendance_df

,Date,TotalStudents,01,01_conf,02,02_conf,03,03_conf,04,04_conf,05,05_conf,06,06_conf,07,07_conf,08,08_conf,09,09_conf
0,12-03-2026,5,1,0.4131,0,0.0,1,0.2466,1,0.6084,0,0.0000,1,0.3509,0,0.0000,0,0.0000,1,0.6282
1,13-02-2026,4,1,0.5627,0,0.0,1,0.3464,1,0.4897,0,0.0000,0,0.0000,0,0.0000,0,0.0000,1,0.6343
2,14-02-2026,2,0,0.0000,0,0.0,0,0.0000,0,0.0000,0,0.0000,0,0.0000,1,0.6881,1,0.7367,0,0.0000
3,15-02-2026,4,1,0.6989,0,0.0,0,0.0000,1,0.6118,1,0.2659,0,0.0000,0,0.0000,0,0.0000,1,0.4805
4,16-02-2026,4,1,0.7667,0,0.0,0,0.0000,1,0.7240,1,0.3484,0,0.0000,0,0.0000,0,0.0000,1,0.6200
5,17-03-2026,4,1,0.7667,0,0.0,0,0.0000,1,0.7240,1,0.3484,0,0.0000,0,0.0000,0,0.0000,1,0.6200
6,18-03-2026,4,1,0.6989,0,0.0,0,0.0000,1,0.6118,1,0.2659,0,0.0000,0,0.0000,0,0.0000,1,0.4805
7,19-03-2026,2,0,0.0000,0,0.0,0,0.0000,0,0.0000,0,0.0000,0,0.0000,1,0.5873,1,0.5177,0,0.0000
8,20-03-2026,2,0,0.0000,0,0.0,0,0.0000,1,0.7714,0,0.0000,1,0.5315,0,0.0000,0,0.0000,0,0.0000
9,23-03-2026,2,0,0.0000,0,0.0,0,0.0000,0,0.0000,0,0.0000,0,0.0000,1,0.5999,1,0.5519,0,0.0000


In [114]:
attendance_df.to_csv(
    CSV_OUTPUT,
    index=False
)

print(
    "CSV Saved:",
    CSV_OUTPUT
)

CSV Saved: attendance.csv


In [115]:
print()

for roll_no in roll_numbers:

    present_count = (
        attendance_df[
            roll_no
        ].sum()
    )

    print(
        f"{roll_no} -> {present_count}"
    )


01 -> 6
02 -> 0
03 -> 2
04 -> 8
05 -> 4
06 -> 3
07 -> 3
08 -> 3
09 -> 6


In [116]:
import cv2
import torch
import onnx
import onnxruntime as ort
import pandas as pd
import numpy as np

print("OpenCV:", cv2.__version__)
print("Torch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ONNXRuntime:", ort.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

OpenCV: 4.13.0
Torch: 2.12.1+cpu
ONNX: 1.22.0
ONNXRuntime: 1.27.0
Pandas: 3.0.3
NumPy: 2.5.0


In [117]:
import tqdm
import matplotlib

print(tqdm.__version__)
print(matplotlib.__version__)

4.68.3
3.11.0
